In [ ]:
!pip install numpy==1.26.2 pandas scikit-learn xgboost --force-reinstall

In [ ]:
!pip freeze

In [ ]:
from PIL import Image
import numpy as np
import os
import cv2
import matplotlib.pyplot as plt
import os
import cv2
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.cluster import BisectingKMeans
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
import re
from xgboost import XGBRegressor
from tqdm import tqdm

In [ ]:
# =======================================================
# 1. CONFIGURAÇÕES E CAMINHOS
# =======================================================
caminhos_planilhas = [
    '/media/workstation_2x_3080/HD 1TB/Famez_yolo/Colorímetro/colorimetro_dezembro_2025.csv',
    '/media/workstation_2x_3080/HD 1TB/Famez_yolo/Colorímetro/colorimetro_fevereiro_2026.csv',
    '/media/workstation_2x_3080/HD 1TB/Famez_yolo/Colorímetro/colorimetro_julho_2026.csv'
]

caminhos_fotos = [
    '/media/workstation_2x_3080/HD 1TB/Famez_yolo/Colorímetro/fotos colorímetro-dezembro-2025',
    '/media/workstation_2x_3080/HD 1TB/Famez_yolo/Colorímetro/Fotos colorímetro- fevereiro-2026',
    '/media/workstation_2x_3080/HD 1TB/Famez_yolo/Colorímetro/Foto colorimetro-Julho2026'
]


In [ ]:
# =======================================================
# 2. CARREGAR E FUNDIR AS PLANILHAS
# =======================================================
def carregar_planilhas(caminhos):
    dfs = []
    for p in caminhos:
        try:
            # Tenta carregar normalmente assumindo que é o formato novo (7 colunas limpas)
            temp = pd.read_csv(p, encoding='utf-8')
            
            # Se for o formato antigo que não tinha cabeçalho correto e tinha 'ID_Seq'
            if len(temp.columns) > 7 or 'ANIMAL' not in temp.columns:
                temp = pd.read_csv(p, encoding='latin-1', header=None, skiprows=2, on_bad_lines='skip')
                temp = temp.iloc[:, :8]
                temp.columns = ['ID_Seq', 'ANIMAL', 'L_pelo', 'a_pelo', 'b_pelo', 'L_pele', 'a_pele', 'b_pele']
                # Remove o ID_Seq para ficar igual ao formato novo (7 colunas)
                temp = temp.drop(columns=['ID_Seq'])
            else:
                # Se for o formato novo, apenas uniformiza os nomes das colunas
                temp.columns = ['ANIMAL', 'L_pelo', 'a_pelo', 'b_pelo', 'L_pele', 'a_pele', 'b_pele']
                
            dfs.append(temp)
        except Exception as e:
            print(f"Erro ao carregar {p}: {e}")
    if not dfs: return pd.DataFrame()
    df_final = pd.concat(dfs, ignore_index=True)
    df_final = df_final.dropna(subset=['ANIMAL'])
    
    # Converte ANIMAL para string limpa
    df_final['ANIMAL'] = df_final['ANIMAL'].astype(float).astype(int).astype(str)
    # Converte colunas LAB para numérico (qualquer erro vira NaN)
    cols_lab = ['L_pelo', 'a_pelo', 'b_pelo', 'L_pele', 'a_pele', 'b_pele']
    for c in cols_lab: 
        df_final[c] = pd.to_numeric(df_final[c], errors='coerce')
    return df_final

In [ ]:
# =======================================================
# 3. CROP E EXTRAÇÃO DA COR DA MANCHA
# =======================================================
def extrair_cor_pura_multicanal(caminho_img):
    img_bgr = cv2.imread(caminho_img)
    if img_bgr is None: return []

    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    max_h, max_w = gray.shape

    # 1. Isolar o círculo
    blur_luz = cv2.GaussianBlur(gray, (61, 61), 0)
    _, thresh_luz = cv2.threshold(blur_luz, 100, 255, cv2.THRESH_BINARY)

    kernel_luz = np.ones((45, 45), np.uint8)
    thresh_luz = cv2.morphologyEx(thresh_luz, cv2.MORPH_OPEN, kernel_luz)

    contornos_luz, _ = cv2.findContours(thresh_luz, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contornos_luz: return []
    maior_contorno_luz = max(contornos_luz, key=cv2.contourArea)

    mascara_amostra = np.zeros_like(gray)
    cv2.drawContours(mascara_amostra, [maior_contorno_luz], -1, 255, thickness=cv2.FILLED)

    # 2. Pintar o cone de branco
    gray_isolado = gray.copy()
    gray_isolado[mascara_amostra == 0] = 255

    # 3. Encontrar a mancha escura (Threshold dinâmico pele/pelo)
    thresh_val = 80 if "pele" in caminho_img.lower() else 165

    blur_mancha = cv2.GaussianBlur(gray_isolado, (151, 151), 0)
    _, mascara_mancha = cv2.threshold(blur_mancha, thresh_val, 255, cv2.THRESH_BINARY_INV)

    kernel_mancha = np.ones((61, 61), np.uint8)
    mascara_mancha = cv2.morphologyEx(mascara_mancha, cv2.MORPH_CLOSE, kernel_mancha)
    mascara_mancha = cv2.morphologyEx(mascara_mancha, cv2.MORPH_OPEN, kernel_mancha)

    contornos_mancha, _ = cv2.findContours(mascara_mancha, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contornos_mancha: return []
    maior_contorno_mancha = max(contornos_mancha, key=cv2.contourArea)

    # Coordenadas da mancha
    X_exato, Y_exato, W_exato, H_exato = cv2.boundingRect(maior_contorno_mancha)

    # 4. Adicionando Borda
    percentual_borda = 0.05
    pad_w = int(W_exato * percentual_borda)
    pad_h = int(H_exato * percentual_borda)

    y1 = max(0, Y_exato - pad_h)
    y2 = min(max_h, Y_exato + H_exato + pad_h)
    x1 = max(0, X_exato - pad_w)
    x2 = min(max_w, X_exato + W_exato + pad_w)

    crop_final = img_bgr[y1:y2, x1:x2]
    if crop_final.size == 0: return []

    crop_rgb = cv2.cvtColor(crop_final, cv2.COLOR_BGR2RGB)
    crop_hsv = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2HSV)
    crop_lab_cv2 = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2LAB)

    # =======================================================
    # MEDIANA E DESVIO PADRÃO
    # =======================================================
    # Usando axis=(0, 1) para calcular a mediana de todos os pixels na Altura e Largura
    rgb_m = np.median(crop_rgb, axis=(0, 1))
    hsv_m = np.median(crop_hsv, axis=(0, 1))
    lab_m = np.median(crop_lab_cv2, axis=(0, 1))

    rgb_s = np.std(crop_rgb, axis=(0, 1))
    hsv_s = np.std(crop_hsv, axis=(0, 1))
    lab_s = np.std(crop_lab_cv2, axis=(0, 1))

    return [{
        'R': rgb_m[0], 'G': rgb_m[1], 'B': rgb_m[2],
        'H': hsv_m[0], 'S': hsv_m[1], 'V': hsv_m[2],
        'L_cv2': lab_m[0], 'a_cv2': lab_m[1], 'b_cv2': lab_m[2],
        'R_std': rgb_s[0], 'G_std': rgb_s[1], 'B_std': rgb_s[2],
        'H_std': hsv_s[0], 'S_std': hsv_s[1], 'V_std': hsv_s[2],
        'L_std': lab_s[0], 'a_std': lab_s[1], 'b_std': lab_s[2]
    }]

In [ ]:
# =======================================================
# 4. PROCESSAR AS FOTOS E MONTAR O DATASET DE COMPARAÇÃO
# =======================================================
print("1. Carregando planilhas...")
df_colorimetro = carregar_planilhas(caminhos_planilhas)
print(f" -> Planilhas carregadas! {len(df_colorimetro)} registros encontrados.")

dataset_final = []
arquivos_sucesso = 0

print("\n2. Processando imagens, extraindo cores e cruzando com a planilha...")

for pasta_atual in caminhos_fotos:
    if not os.path.exists(pasta_atual):
        print(f" -> Aviso: Pasta não encontrada: {pasta_atual}")
        continue

    for arquivo in os.listdir(pasta_atual):
        if not arquivo.lower().endswith(('.jpg', '.png', '.jpeg')): continue

        caminho_completo = os.path.join(pasta_atual, arquivo)

        match = re.search(r'\d+', arquivo)
        if match:
            animal_id = match.group()
        else:
            continue

        tipo = "pele" if "pele" in arquivo.lower() else "pelo"

        linha = df_colorimetro[df_colorimetro['ANIMAL'] == animal_id]
        if linha.empty: continue

        if tipo == "pele":
            l, a, b = linha[['pele_L', 'pele_a', 'pele_b']].iloc[0]
        else:
            l, a, b = linha[['pelo_L', 'pelo_a', 'pelo_b']].iloc[0]

        if pd.isna(l): continue

        features_lista = extrair_cor_pura_multicanal(caminho_completo)
        if not features_lista: continue

        for f in features_lista:
            f['Animal'] = animal_id
            f['Arquivo'] = arquivo
            f['Tipo_Amostra'] = tipo

            f['L_Colorimetro'] = l
            f['a_Colorimetro'] = a
            f['b_Colorimetro'] = b
            dataset_final.append(f)

        arquivos_sucesso += 1

df_comparacao = pd.DataFrame(dataset_final)
print(f" -> Sucesso: {arquivos_sucesso} fotos processadas e alinhadas com a planilha.")

# =======================================================
# 5. CÁLCULO DE ERRO E DELTA E (IMAGEM VS PLANILHA)
# =======================================================
if len(df_comparacao) > 0:
    print("\n3. Calculando a margem de erro...")
    df_comparacao['L_img_convertido'] = df_comparacao['L_cv2'] * (100.0 / 255.0)
    df_comparacao['a_img_convertido'] = df_comparacao['a_cv2'] - 128.0
    df_comparacao['b_img_convertido'] = df_comparacao['b_cv2'] - 128.0

    df_comparacao['Erro_L'] = abs(df_comparacao['L_img_convertido'] - df_comparacao['L_Colorimetro'])
    df_comparacao['Erro_a'] = abs(df_comparacao['a_img_convertido'] - df_comparacao['a_Colorimetro'])
    df_comparacao['Erro_b'] = abs(df_comparacao['b_img_convertido'] - df_comparacao['b_Colorimetro'])

    df_comparacao['Delta_E'] = np.sqrt(
        (df_comparacao['L_img_convertido'] - df_comparacao['L_Colorimetro'])**2 +
        (df_comparacao['a_img_convertido'] - df_comparacao['a_Colorimetro'])**2 +
        (df_comparacao['b_img_convertido'] - df_comparacao['b_Colorimetro'])**2
    )

    print("\n=======================================================")
    print("      RESULTADOS FINAIS (FOTO vs EQUIPAMENTO)      ")
    print("=======================================================")
    print(f"Diferença Média L* (Luminosidade): ± {df_comparacao['Erro_L'].mean():.2f}")
    print(f"Diferença Média a* (Verde-Vermelho): ± {df_comparacao['Erro_a'].mean():.2f}")
    print(f"Diferença Média b* (Azul-Amarelo): ± {df_comparacao['Erro_b'].mean():.2f}")
    print("-------------------------------------------------------")
    print(f"Delta E Médio (Distância Geral de Cor): {df_comparacao['Delta_E'].mean():.2f}")
    print("=======================================================\n")

    colunas_exibir = ['Animal', 'Tipo_Amostra', 'L_img_convertido', 'L_Colorimetro', 'Erro_L', 'Delta_E']
    print("Amostra dos dados comparados:")
    print(df_comparacao[colunas_exibir].head().to_string())
    df_comparacao.to_csv("resultados.csv", index=False)
else:
    pass


In [ ]:
# =======================================================
# EXTRAÇÃO MULTICANAL EM GRID (PARA ML)
# =======================================================
def extrair_grid_multicanal(caminho_img, num_partes=9):
    img_bgr = cv2.imread(caminho_img)
    if img_bgr is None: return []
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    max_h, max_w = gray.shape
    blur_luz = cv2.GaussianBlur(gray, (61, 61), 0)
    _, thresh_luz = cv2.threshold(blur_luz, 100, 255, cv2.THRESH_BINARY)
    kernel_luz = np.ones((45, 45), np.uint8)
    thresh_luz = cv2.morphologyEx(thresh_luz, cv2.MORPH_OPEN, kernel_luz)
    contornos_luz, _ = cv2.findContours(thresh_luz, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contornos_luz: return []
    maior_contorno_luz = max(contornos_luz, key=cv2.contourArea)
    mascara_amostra = np.zeros_like(gray)
    cv2.drawContours(mascara_amostra, [maior_contorno_luz], -1, 255, thickness=cv2.FILLED)
    gray_isolado = gray.copy()
    gray_isolado[mascara_amostra == 0] = 255
    thresh_val = 80 if "pele" in caminho_img.lower() else 165
    blur_mancha = cv2.GaussianBlur(gray_isolado, (151, 151), 0)
    _, mascara_mancha = cv2.threshold(blur_mancha, thresh_val, 255, cv2.THRESH_BINARY_INV)
    kernel_mancha = np.ones((61, 61), np.uint8)
    mascara_mancha = cv2.morphologyEx(mascara_mancha, cv2.MORPH_CLOSE, kernel_mancha)
    mascara_mancha = cv2.morphologyEx(mascara_mancha, cv2.MORPH_OPEN, kernel_mancha)
    contornos_mancha, _ = cv2.findContours(mascara_mancha, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contornos_mancha: return []
    maior_contorno_mancha = max(contornos_mancha, key=cv2.contourArea)
    X_exato, Y_exato, W_exato, H_exato = cv2.boundingRect(maior_contorno_mancha)
    percentual_borda = 0.05
    pad_w = int(W_exato * percentual_borda)
    pad_h = int(H_exato * percentual_borda)
    y1 = max(0, Y_exato - pad_h)
    y2 = min(max_h, Y_exato + H_exato + pad_h)
    x1 = max(0, X_exato - pad_w)
    x2 = min(max_w, X_exato + W_exato + pad_w)
    crop_final = img_bgr[y1:y2, x1:x2]
    if crop_final.size == 0: return []
    crop_rgb = cv2.cvtColor(crop_final, cv2.COLOR_BGR2RGB)
    crop_hsv = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2HSV)
    crop_lab_cv2 = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2LAB)
    grid = int(np.sqrt(num_partes))
    h_c, w_c = crop_rgb.shape[:2]
    step_h, step_w = h_c // grid, w_c // grid
    if step_h == 0 or step_w == 0: return []
    lista_features = []
    for i in range(grid):
        for j in range(grid):
            y_inicio = i * step_h
            y_fim = (i + 1) * step_h if i < grid - 1 else h_c
            x_inicio = j * step_w
            x_fim = (j + 1) * step_w if j < grid - 1 else w_c
            bloco_rgb = crop_rgb[y_inicio:y_fim, x_inicio:x_fim]
            bloco_hsv = crop_hsv[y_inicio:y_fim, x_inicio:x_fim]
            bloco_lab = crop_lab_cv2[y_inicio:y_fim, x_inicio:x_fim]
            rgb_m = np.mean(bloco_rgb, axis=(0,1))
            hsv_m = np.mean(bloco_hsv, axis=(0,1))
            lab_m = np.mean(bloco_lab, axis=(0,1))
            lista_features.append({
                'R': rgb_m[0], 'G': rgb_m[1], 'B': rgb_m[2],
                'H': hsv_m[0], 'S': hsv_m[1], 'V': hsv_m[2],
                'L_cv2': lab_m[0], 'a_cv2': lab_m[1], 'b_cv2': lab_m[2]
            })
    return lista_features


In [ ]:
# =======================================================
# 7. MONTAR O DATASET DE MACHINE LEARNING
# =======================================================
print("1. Carregando planilhas...")
df_colorimetro = carregar_planilhas(caminhos_planilhas)

agrupamento = {}
for pasta_atual in caminhos_fotos:
    if not os.path.exists(pasta_atual): continue
    for arquivo in os.listdir(pasta_atual):
        if not arquivo.lower().endswith(('.jpg', '.png', '.jpeg')): continue
        match = re.search(r'\d+', arquivo)
        if not match: continue
        animal_id = match.group()
        tipo = "pele" if "pele" in arquivo.lower() else "pelo"
        
        if animal_id not in agrupamento:
            agrupamento[animal_id] = {'pele': None, 'pelo': None}
        
        caminho_completo = os.path.join(pasta_atual, arquivo)
        agrupamento[animal_id][tipo] = caminho_completo

dataset_final = []
from tqdm import tqdm
import pandas as pd
for animal_id, imgs in tqdm(agrupamento.items(), desc="Processando pares"):
    linha = df_colorimetro[df_colorimetro['ANIMAL'] == animal_id]
    if linha.empty: continue
    
    if imgs['pele']:
        l_pele, a_pele, b_pele = linha[['pele_L', 'pele_a', 'pele_b']].iloc[0]
        if not pd.isna(l_pele):
            features_lista = extrair_grid_multicanal(imgs['pele'])
            for f in features_lista:
                f.update({'ANIMAL': animal_id, 'L': l_pele, 'a': a_pele, 'b': b_pele, 'Tipo_Pele': 1})
                dataset_final.append(f)
                
    if imgs['pelo']:
        l_pelo, a_pelo, b_pelo = linha[['pelo_L', 'pelo_a', 'pelo_b']].iloc[0]
        if not pd.isna(l_pelo):
            features_lista = extrair_grid_multicanal(imgs['pelo'])
            for f in features_lista:
                f.update({'ANIMAL': animal_id, 'L': l_pelo, 'a': a_pelo, 'b': b_pelo, 'Tipo_Pele': 0})
                dataset_final.append(f)

df_ml = pd.DataFrame(dataset_final)
print(f"\nDataset ML montado com {len(df_ml)} blocos.")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# =======================================================
# ANÁLISE DE CORRELAÇÃO (GRÁFICOS)
# =======================================================
print("Gerando gráficos de correlação...")

# Limpar dados nulos caso tenha passado algum
df_plot = df_ml.dropna(subset=['R', 'G', 'B', 'L', 'a', 'b', 'L_cv2', 'a_cv2', 'b_cv2'])

# Selecionamos apenas as colunas numéricas de cor
colunas_cor = ['R', 'G', 'B', 'L_cv2', 'a_cv2', 'b_cv2', 'L', 'a', 'b']
matriz_corr = df_plot[colunas_cor].corr()

plt.figure(figsize=(10, 8))
# Cria um mapa de calor
sns.heatmap(matriz_corr, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title("Matriz de Correlação: RGB vs LAB (OpenCV) vs LAB (Colorímetro)")
plt.tight_layout()
plt.show()

# Gera uma grade de gráficos cruzando R, G e B com L, a, b reais
# O kind='reg' desenha uma regressão linear
g = sns.pairplot(
    df_plot,
    x_vars=['R', 'G', 'B'],
    y_vars=['L', 'a', 'b'],
    hue='Tipo_Pele',
    kind='scatter',
    plot_kws={'alpha': 0.6}
)
g.fig.suptitle("Dispersão: RGB da Câmera vs LAB Real do Colorímetro", y=1.05)
plt.show()

In [ ]:
# Muito útil para ver se a conversão matemática da imagem está próxima da realidade
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sns.regplot(data=df_plot, x='L_cv2', y='L', ax=axes[0], scatter_kws={'alpha':0.5})
axes[0].set_title('Luminosidade (L): OpenCV vs Real')

sns.regplot(data=df_plot, x='a_cv2', y='a', ax=axes[1], scatter_kws={'alpha':0.5}, color='green')
axes[1].set_title('Eixo Verde-Vermelho (a): OpenCV vs Real')

sns.regplot(data=df_plot, x='b_cv2', y='b', ax=axes[2], scatter_kws={'alpha':0.5}, color='orange')
axes[2].set_title('Eixo Azul-Amarelo (b): OpenCV vs Real')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from scipy.stats import pearsonr
# =======================================================
# 5. TREINAMENTO COM K-MEANS E EXTRA TREES
# =======================================================
# Consertando o nome da coluna caso tenha vindo maiúsculo do bloco 4
df_ml = df_ml.rename(columns={'ANIMAL': 'Animal'})

if len(df_ml) > 10:
    print("\n3. Iniciando Clusterização Global do Dataset...")

    colunas_cor = ['R', 'G', 'B', 'H', 'S', 'V', 'L_cv2', 'a_cv2', 'b_cv2']
    n_clusters = 9

    kmeans_global = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')

    df_ml['Familia_KMeans'] = kmeans_global.fit_predict(df_ml[colunas_cor])
    print(f" -> K-Means finalizou o agrupamento em {n_clusters} Famílias.")


    # Agora a IA recebe as cores, se é pele/pelo, E a qual cluster aquele recorte pertence
    colunas_x = ['R', 'G', 'B', 'H', 'S', 'V', 'L_cv2', 'a_cv2', 'b_cv2', 'Tipo_Pele', 'Familia_KMeans']

    X = df_ml[colunas_x]
    y = df_ml[['L', 'a', 'b']]
    grupos = df_ml['Animal']

    separador = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    indice_treino, indice_teste = next(separador.split(X, y, groups=grupos))

    X_train, X_test = X.iloc[indice_treino], X.iloc[indice_teste]
    y_train, y_test = y.iloc[indice_treino], y.iloc[indice_teste]

    print(f"\n4. Calibrando Extra Trees usando K-Means...")
    print(f" -> Treino com {df_ml.iloc[indice_treino]['Animal'].nunique()} animais.")
    print(f" -> Teste {X_test.shape[0]} amostras de {df_ml.iloc[indice_teste]['Animal'].nunique()} animais DESCONHECIDOS.")

    modelo = ExtraTreesRegressor(n_estimators=300, max_depth=20, random_state=42)
    modelo.fit(X_train, y_train)

    previsoes = modelo.predict(X_test)
    prev_L = previsoes[:,0]
    prev_a = previsoes[:,1]
    prev_b = previsoes[:,2]

    y_L = y_test['L'].values
    y_a = y_test['a'].values
    y_b = y_test['b'].values

    mae_L = mean_absolute_error(y_L, prev_L)
    mae_a = mean_absolute_error(y_a, prev_a)
    mae_b = mean_absolute_error(y_b, prev_b)

    r2_L = r2_score(y_L, prev_L)
    r2_a = r2_score(y_a, prev_a)
    r2_b = r2_score(y_b, prev_b)

    rmse_L = np.sqrt(mean_squared_error(y_L, prev_L))
    rmse_a = np.sqrt(mean_squared_error(y_a, prev_a))
    rmse_b = np.sqrt(mean_squared_error(y_b, prev_b))

    from scipy.stats import pearsonr
    pearson_L, _ = pearsonr(y_L, prev_L)
    pearson_a, _ = pearsonr(y_a, prev_a)
    pearson_b, _ = pearsonr(y_b, prev_b)

    delta_e = np.sqrt((y_L - prev_L)**2 + (y_a - prev_a)**2 + (y_b - prev_b)**2)
    delta_e_mean = np.mean(delta_e)

    bias_L = np.mean(prev_L - y_L)
    sd_L = np.std(prev_L - y_L)
    loa_L = (bias_L - 1.96*sd_L, bias_L + 1.96*sd_L)

    bias_a = np.mean(prev_a - y_a)
    sd_a = np.std(prev_a - y_a)
    loa_a = (bias_a - 1.96*sd_a, bias_a + 1.96*sd_a)

    bias_b = np.mean(prev_b - y_b)
    sd_b = np.std(prev_b - y_b)
    loa_b = (bias_b - 1.96*sd_b, bias_b + 1.96*sd_b)

    print("\n=======================================================")
    print("             MÉTRICAS DE AVALIAÇÃO (CONJUNTO DE TESTE) ")
    print("=======================================================")
    print(f"Erro Médio Absoluto (MAE): L*={mae_L:.3f} | a*={mae_a:.3f} | b*={mae_b:.3f}")
    print(f"Coeficiente de Determinação (R2): L*={r2_L:.3f} | a*={r2_a:.3f} | b*={r2_b:.3f}")
    print(f"Raiz do Erro Quadrático Médio (RMSE): L*={rmse_L:.3f} | a*={rmse_a:.3f} | b*={rmse_b:.3f}")
    print(f"Coeficiente de Correlação (r): L*={pearson_L:.3f} | a*={pearson_a:.3f} | b*={pearson_b:.3f}")
    print(f"Diferença de Cor (Delta E) Médio: {delta_e_mean:.3f}")
    print("\n--- ANÁLISE DE BLAND-ALTMAN ---")
    print(f"L* -> Viés: {bias_L:.3f} | Limites de Concordância 95%: ({loa_L[0]:.3f}, {loa_L[1]:.3f})")
    print(f"a* -> Viés: {bias_a:.3f} | Limites de Concordância 95%: ({loa_a[0]:.3f}, {loa_a[1]:.3f})")
    print(f"b* -> Viés: {bias_b:.3f} | Limites de Concordância 95%: ({loa_b[0]:.3f}, {loa_b[1]:.3f})")
    print("=======================================================")
else:
    print("\n[Erro] Não há dados suficientes para treinar a IA.")



In [ ]:
# =======================================================
# 5. TREINAMENTO COM RANDOM FOREST (DUPLO ESPECIALISTA)
# =======================================================
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from scipy.stats import pearsonr
import pandas as pd
import numpy as np

if len(df_ml) > 10:
    print("\n3. Aplicando Engenharia de Features e Treinando Especialistas (PELE e PELO)...")
    df_ia = df_ml.dropna().copy()

    # ==========================================
    # ESTRATÉGIA: CRIANDO NOVAS FEATURES MATEMÁTICAS
    # ==========================================
    df_ia['Razao_RG'] = df_ia['R'] / (df_ia['G'] + 0.000001)
    df_ia['Razao_RB'] = df_ia['R'] / (df_ia['B'] + 0.000001)
    df_ia['Razao_GB'] = df_ia['G'] / (df_ia['B'] + 0.000001)
    df_ia['Dif_L_V'] = df_ia['L_cv2'] - df_ia['V']
    df_ia['Soma_RGB'] = df_ia['R'] + df_ia['G'] + df_ia['B']

    features = [
        'R', 'G', 'B', 'H', 'S', 'V', 'L_cv2', 'a_cv2', 'b_cv2',
        'Razao_RG', 'Razao_RB', 'Razao_GB', 'Dif_L_V', 'Soma_RGB'
    ]
    targets = ['L', 'a', 'b']

    # ==========================================
    # SEPARAÇÃO TREINO / TESTE POR ANIMAL (Evita Vazamento de Dados)
    # ==========================================
    # Pega o nome correto da coluna (garante que funciona independentemente de ser ANIMAL ou Animal)
    col_animal = 'ANIMAL' if 'ANIMAL' in df_ia.columns else 'Animal'
    grupos = df_ia[col_animal]

    separador = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    indice_treino, indice_teste = next(separador.split(df_ia[features], df_ia[targets], groups=grupos))

    df_treino = df_ia.iloc[indice_treino].copy()
    df_teste = df_ia.iloc[indice_teste].copy()

    print(f" -> Ensinando a IA com amostras de {df_treino[col_animal].nunique()} animais.")
    print(f" -> Testando a IA com amostras de {df_teste[col_animal].nunique()} animais DESCONHECIDOS.")

    # ==========================================
    # MODELO 1: ESPECIALISTA EM PELE
    # ==========================================
    treino_pele = df_treino[df_treino['Tipo_Pele'] == 1]
    teste_pele = df_teste[df_teste['Tipo_Pele'] == 1]

    rf_pele = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
    if len(treino_pele) > 0:
        rf_pele.fit(treino_pele[features], treino_pele[targets])
        prev_pele = pd.DataFrame(rf_pele.predict(teste_pele[features]), columns=['L_pred', 'a_pred', 'b_pred'])
        Y_test_pele = teste_pele[targets].reset_index(drop=True)
    else:
        prev_pele = pd.DataFrame(columns=['L_pred', 'a_pred', 'b_pred'])
        Y_test_pele = pd.DataFrame(columns=targets)

    # ==========================================
    # MODELO 2: ESPECIALISTA EM PELO
    # ==========================================
    treino_pelo = df_treino[df_treino['Tipo_Pele'] == 0]
    teste_pelo = df_teste[df_teste['Tipo_Pele'] == 0]

    rf_pelo = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
    if len(treino_pelo) > 0:
        rf_pelo.fit(treino_pelo[features], treino_pelo[targets])
        prev_pelo = pd.DataFrame(rf_pelo.predict(teste_pelo[features]), columns=['L_pred', 'a_pred', 'b_pred'])
        Y_test_pelo = teste_pelo[targets].reset_index(drop=True)
    else:
        prev_pelo = pd.DataFrame(columns=['L_pred', 'a_pred', 'b_pred'])
        Y_test_pelo = pd.DataFrame(columns=targets)

    # ==========================================
    # AVALIAÇÃO E RESULTADOS
    # ==========================================
    # Concatena as validações de pele e pelo para fazer as métricas globais combinadas!
    Y_test_geral = pd.concat([Y_test_pele, Y_test_pelo]).reset_index(drop=True)
    prev_geral = pd.concat([prev_pele, prev_pelo]).reset_index(drop=True)

    y_L = Y_test_geral['L'].values
    y_a = Y_test_geral['a'].values
    y_b = Y_test_geral['b'].values

    prev_L = prev_geral['L_pred'].values
    prev_a = prev_geral['a_pred'].values
    prev_b = prev_geral['b_pred'].values

    # Cálculos Matemáticos
    mae_L = mean_absolute_error(y_L, prev_L)
    mae_a = mean_absolute_error(y_a, prev_a)
    mae_b = mean_absolute_error(y_b, prev_b)

    r2_L = r2_score(y_L, prev_L)
    r2_a = r2_score(y_a, prev_a)
    r2_b = r2_score(y_b, prev_b)

    rmse_L = np.sqrt(mean_squared_error(y_L, prev_L))
    rmse_a = np.sqrt(mean_squared_error(y_a, prev_a))
    rmse_b = np.sqrt(mean_squared_error(y_b, prev_b))

    pearson_L, _ = pearsonr(y_L, prev_L)
    pearson_a, _ = pearsonr(y_a, prev_a)
    pearson_b, _ = pearsonr(y_b, prev_b)

    delta_e = np.sqrt((y_L - prev_L)**2 + (y_a - prev_a)**2 + (y_b - prev_b)**2)
    delta_e_mean = np.mean(delta_e)

    bias_L = np.mean(prev_L - y_L)
    sd_L = np.std(prev_L - y_L)
    loa_L = (bias_L - 1.96*sd_L, bias_L + 1.96*sd_L)

    bias_a = np.mean(prev_a - y_a)
    sd_a = np.std(prev_a - y_a)
    loa_a = (bias_a - 1.96*sd_a, bias_a + 1.96*sd_a)

    bias_b = np.mean(prev_b - y_b)
    sd_b = np.std(prev_b - y_b)
    loa_b = (bias_b - 1.96*sd_b, bias_b + 1.96*sd_b)

    # =======================================================
    # IMPRESSÃO DO RESULTADO
    # =======================================================
    print("\n=======================================================")
    print("             MÉTRICAS DE AVALIAÇÃO (CONJUNTO DE TESTE) ")
    print("=======================================================")
    print(f"Erro Médio Absoluto (MAE): L*={mae_L:.3f} | a*={mae_a:.3f} | b*={mae_b:.3f}")
    print(f"Coeficiente de Determinação (R2): L*={r2_L:.3f} | a*={r2_a:.3f} | b*={r2_b:.3f}")
    print(f"Raiz do Erro Quadrático Médio (RMSE): L*={rmse_L:.3f} | a*={rmse_a:.3f} | b*={rmse_b:.3f}")
    print(f"Coeficiente de Correlação (r): L*={pearson_L:.3f} | a*={pearson_a:.3f} | b*={pearson_b:.3f}")
    print(f"Diferença de Cor (Delta E) Médio: {delta_e_mean:.3f}")
    print("\n--- ANÁLISE DE BLAND-ALTMAN ---")
    print(f"L* -> Viés: {bias_L:.3f} | Limites de Concordância 95%: ({loa_L[0]:.3f}, {loa_L[1]:.3f})")
    print(f"a* -> Viés: {bias_a:.3f} | Limites de Concordância 95%: ({loa_a[0]:.3f}, {loa_a[1]:.3f})")
    print(f"b* -> Viés: {bias_b:.3f} | Limites de Concordância 95%: ({loa_b[0]:.3f}, {loa_b[1]:.3f})")
    print("=======================================================")
else:
    print("\n[Erro] Não há dados suficientes para treinar a IA.")


In [ ]:
# =======================================================
# 5. TREINAMENTO COM EXTRA TREES
# =======================================================
if len(df_ml) > 10:
    print("\n3. Iniciando Treinamento da Inteligência Artificial (Extra Trees)...")

    # Usando as 10 variáveis
    #colunas_x= [
    #    'R', 'G', 'B', 'H', 'S', 'V', 'L_cv2', 'a_cv2', 'b_cv2',
    #    'R_std', 'G_std', 'B_std', 'H_std', 'S_std', 'V_std', 'L_std', 'a_std', 'b_std']

   #colunas_x= [ 'R', 'G', 'B', 'H', 'S', 'V', 'L_cv2', 'a_cv2', 'b_cv2']
    colunas_x = ['R', 'G', 'B', 'H', 'S', 'V', 'L_cv2', 'a_cv2', 'b_cv2', 'Tipo_Pele']
    X = df_ml[colunas_x]
    y = df_ml[['L', 'a', 'b']]
    grupos = df_ml['Animal']

    separador = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    indice_treino, indice_teste = next(separador.split(X, y, groups=grupos))

    X_train, X_test = X.iloc[indice_treino], X.iloc[indice_teste]
    y_train, y_test = y.iloc[indice_treino], y.iloc[indice_teste]

    print(f" -> Ensinando a IA com {X_train.shape[0]} amostras de {df_ml.iloc[indice_treino]['Animal'].nunique()} animais.")
    print(f" -> Testando a IA com {X_test.shape[0]} amostras de {df_ml.iloc[indice_teste]['Animal'].nunique()} animais DESCONHECIDOS.")

    modelo = ExtraTreesRegressor(n_estimators=300, max_depth=20, random_state=42)
    modelo.fit(X_train, y_train)

    previsoes = modelo.predict(X_test)
    prev_L = previsoes[:,0]
    prev_a = previsoes[:,1]
    prev_b = previsoes[:,2]

    y_L = y_test['L'].values
    y_a = y_test['a'].values
    y_b = y_test['b'].values

    mae_L = mean_absolute_error(y_L, prev_L)
    mae_a = mean_absolute_error(y_a, prev_a)
    mae_b = mean_absolute_error(y_b, prev_b)

    r2_L = r2_score(y_L, prev_L)
    r2_a = r2_score(y_a, prev_a)
    r2_b = r2_score(y_b, prev_b)

    rmse_L = np.sqrt(mean_squared_error(y_L, prev_L))
    rmse_a = np.sqrt(mean_squared_error(y_a, prev_a))
    rmse_b = np.sqrt(mean_squared_error(y_b, prev_b))

    from scipy.stats import pearsonr
    pearson_L, _ = pearsonr(y_L, prev_L)
    pearson_a, _ = pearsonr(y_a, prev_a)
    pearson_b, _ = pearsonr(y_b, prev_b)

    delta_e = np.sqrt((y_L - prev_L)**2 + (y_a - prev_a)**2 + (y_b - prev_b)**2)
    delta_e_mean = np.mean(delta_e)

    bias_L = np.mean(prev_L - y_L)
    sd_L = np.std(prev_L - y_L)
    loa_L = (bias_L - 1.96*sd_L, bias_L + 1.96*sd_L)

    bias_a = np.mean(prev_a - y_a)
    sd_a = np.std(prev_a - y_a)
    loa_a = (bias_a - 1.96*sd_a, bias_a + 1.96*sd_a)

    bias_b = np.mean(prev_b - y_b)
    sd_b = np.std(prev_b - y_b)
    loa_b = (bias_b - 1.96*sd_b, bias_b + 1.96*sd_b)

    print("\n=======================================================")
    print("             MÉTRICAS DE AVALIAÇÃO (CONJUNTO DE TESTE) ")
    print("=======================================================")
    print(f"Erro Médio Absoluto (MAE): L*={mae_L:.3f} | a*={mae_a:.3f} | b*={mae_b:.3f}")
    print(f"Coeficiente de Determinação (R2): L*={r2_L:.3f} | a*={r2_a:.3f} | b*={r2_b:.3f}")
    print(f"Raiz do Erro Quadrático Médio (RMSE): L*={rmse_L:.3f} | a*={rmse_a:.3f} | b*={rmse_b:.3f}")
    print(f"Coeficiente de Correlação (r): L*={pearson_L:.3f} | a*={pearson_a:.3f} | b*={pearson_b:.3f}")
    print(f"Diferença de Cor (Delta E) Médio: {delta_e_mean:.3f}")
    print("\n--- ANÁLISE DE BLAND-ALTMAN ---")
    print(f"L* -> Viés: {bias_L:.3f} | Limites de Concordância 95%: ({loa_L[0]:.3f}, {loa_L[1]:.3f})")
    print(f"a* -> Viés: {bias_a:.3f} | Limites de Concordância 95%: ({loa_a[0]:.3f}, {loa_a[1]:.3f})")
    print(f"b* -> Viés: {bias_b:.3f} | Limites de Concordância 95%: ({loa_b[0]:.3f}, {loa_b[1]:.3f})")
    print("=======================================================")
else:
    print("\n[Erro] Não há dados suficientes para treinar a IA.")



In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

if len(df_ml) > 0:
    colunas_x = ['R', 'G', 'B', 'H', 'S', 'V', 'L_cv2', 'a_cv2', 'b_cv2', 'Tipo_Pele']
    X = df_ml[colunas_x]
    y = df_ml[['L', 'a', 'b']]
    grupos = df_ml['Animal']

    separador = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    indice_treino, indice_teste = next(separador.split(X, y, groups=grupos))

    X_train, X_test = X.iloc[indice_treino], X.iloc[indice_teste]
    y_train, y_test = y.iloc[indice_treino], y.iloc[indice_teste]

    config_xgb = {
        'n_estimators': 600,
        'learning_rate': 0.05,
        'max_depth': 20,
        'random_state': 42,
        'tree_method': 'hist',
        'device': 'cpu'
    }

    modelo_L = Pipeline([('scaler', StandardScaler()), ('xgb', XGBRegressor(**config_xgb))])
    modelo_a = Pipeline([('scaler', StandardScaler()), ('xgb', XGBRegressor(**config_xgb))])
    modelo_b = Pipeline([('scaler', StandardScaler()), ('xgb', XGBRegressor(**config_xgb))])

    print("\n -> Treinando especialista L*...")
    modelo_L.fit(X_train, y_train['L'])
    print(" -> Treinando especialista a*...")
    modelo_a.fit(X_train, y_train['a'])
    print(" -> Treinando especialista b*...")
    modelo_b.fit(X_train, y_train['b'])

    prev_L = modelo_L.predict(X_test)
    prev_a = modelo_a.predict(X_test)
    prev_b = modelo_b.predict(X_test)

    y_L = y_test['L'].values
    y_a = y_test['a'].values
    y_b = y_test['b'].values

    mae_L = mean_absolute_error(y_L, prev_L)
    mae_a = mean_absolute_error(y_a, prev_a)
    mae_b = mean_absolute_error(y_b, prev_b)

    r2_L = r2_score(y_L, prev_L)
    r2_a = r2_score(y_a, prev_a)
    r2_b = r2_score(y_b, prev_b)

    rmse_L = np.sqrt(mean_squared_error(y_L, prev_L))
    rmse_a = np.sqrt(mean_squared_error(y_a, prev_a))
    rmse_b = np.sqrt(mean_squared_error(y_b, prev_b))

    pearson_L, _ = pearsonr(y_L, prev_L)
    pearson_a, _ = pearsonr(y_a, prev_a)
    pearson_b, _ = pearsonr(y_b, prev_b)

    delta_e = np.sqrt((y_L - prev_L)**2 + (y_a - prev_a)**2 + (y_b - prev_b)**2)
    delta_e_mean = np.mean(delta_e)

    bias_L = np.mean(prev_L - y_L)
    sd_L = np.std(prev_L - y_L)
    loa_L = (bias_L - 1.96*sd_L, bias_L + 1.96*sd_L)

    bias_a = np.mean(prev_a - y_a)
    sd_a = np.std(prev_a - y_a)
    loa_a = (bias_a - 1.96*sd_a, bias_a + 1.96*sd_a)

    bias_b = np.mean(prev_b - y_b)
    sd_b = np.std(prev_b - y_b)
    loa_b = (bias_b - 1.96*sd_b, bias_b + 1.96*sd_b)

    print("\n=======================================================")
    print("             MÉTRICAS DE AVALIAÇÃO (CONJUNTO DE TESTE) ")
    print("=======================================================")
    print(f"Erro Médio Absoluto (MAE): L*={mae_L:.3f} | a*={mae_a:.3f} | b*={mae_b:.3f}")
    print(f"Coeficiente de Determinação (R2): L*={r2_L:.3f} | a*={r2_a:.3f} | b*={r2_b:.3f}")
    print(f"Raiz do Erro Quadrático Médio (RMSE): L*={rmse_L:.3f} | a*={rmse_a:.3f} | b*={rmse_b:.3f}")
    print(f"Coeficiente de Correlação (r): L*={pearson_L:.3f} | a*={pearson_a:.3f} | b*={pearson_b:.3f}")
    print(f"Diferença de Cor (Delta E) Médio: {delta_e_mean:.3f}")
    print("\n--- ANÁLISE DE BLAND-ALTMAN ---")
    print(f"L* -> Viés: {bias_L:.3f} | Limites de Concordância 95%: ({loa_L[0]:.3f}, {loa_L[1]:.3f})")
    print(f"a* -> Viés: {bias_a:.3f} | Limites de Concordância 95%: ({loa_a[0]:.3f}, {loa_a[1]:.3f})")
    print(f"b* -> Viés: {bias_b:.3f} | Limites de Concordância 95%: ({loa_b[0]:.3f}, {loa_b[1]:.3f})")
    print("=======================================================")
else:
    print("Nenhum dado encontrado para treino.")
